# Hybrid structural + machine-learning bankruptcy prediction: implementation

**Question.** Do Merton-type structural features help machine-learning models predict US corporate bankruptcy, especially when data are scarce or the world shifts (new period, new industry)?

**Data.** Every 10-K/10-Q filing of non-financial SEC registrants, 2010–2025 (252,943 filings, 10,743 firms). The target `y = 1` if the firm files an 8-K Item 1.03 (bankruptcy) within 12 months after the filing date.

**Plan of the notebook**
1. Data overview
2. Structural model (Merton DD from SEC data) on its own
3. Equal-budget hyper-parameter tuning
4. ML-only models: logit, random forest, XGB-static, XGB-lag, LSTM
5. Hybrid models: H1 features, H2 structural prior, H3 stacking
6. Main out-of-time results with confidence intervals
7. What drives the predictions (importance, sector and size)
8. Evaluation regimes: rolling years, out-of-industry, unseen firms, limited data, horizon
9. Ablation studies
10. Summary of saved outputs

Run from the repository root. Figures go to `results/figures/`, tables to `results/tables/`, predictions to `results/predictions/`.

## 0. Setup

The switches below control the run time. With the defaults, the notebook takes roughly 1–3 hours on a laptop CPU: tuning (random forest and the two XGBoost families, ~15–30 min each), the LSTMs and the limited-data repeats. For a quick first pass set `TUNE_TRIALS = 0`, `RUN_LSTM = False`, `LIMITED_DATA_REPEATS = 2`.

In [ ]:
import json
import time
import warnings
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from src.pipeline import (MODELS, FEATURE_SETS, GROUPS, TARGET, backtest_buckets, describe_panel,
                          feature_columns, firm_holdout_split, fit_predict, industry_split, load_panel,
                          oot_split, paired_bootstrap_delta, permutation_importance, plots, relabel,
                          results_table, rolling_splits, stack, structural_prior,
                          subsample_training_firms, tune)
from sklearn.metrics import average_precision_score, roc_auc_score

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)

# ---- run-time switches ----
TUNE_TRIALS = 4             # random-search trials per model family, incl. the default (0 = defaults only); the reported run used 4
RUN_LSTM = True             # LSTM experiments (slowest part on CPU)
LSTM_SEEDS = (0, 1, 2)      # LSTM = average of these seeds
LSTM_TUNE_TRIALS = 0        # LSTM random-search trials (expensive; 0 = defaults)
BOOTSTRAP = 500             # firm-bootstrap replicates for confidence intervals
LIMITED_DATA_REPEATS = 5    # repeats per training fraction in the limited-data study

RESULTS = Path("results")
FIG, TAB, PRED = RESULTS / "figures", RESULTS / "tables", RESULTS / "predictions"
for folder in (FIG, TAB, PRED):
    folder.mkdir(parents=True, exist_ok=True)


def save_table(df, name, show=True):
    df.to_csv(TAB / f"{name}.csv")
    if show:
        display(df.round(4))
    return df

## 1. Data

One row is one filing. Every feature uses only information public on the filing date. The main split is out-of-time with a one-year gap on each side, because labels look 12 months ahead:

| Train | Gap | Validation | Gap | Test |
|---|---|---|---|---|
| 2010–2018 | 2019 | 2020 | 2021 | 2022–Oct 2025 |

Validation is used only for calibration and early stopping, never for choosing models by test performance.

In [ ]:
frame = load_panel()        # also builds the lag (history) features
split = oot_split(frame)
print(f"{len(frame):,} filings, {frame.firm.nunique():,} firms, {frame.y.sum():,} positive filings")

by_year = save_table(describe_panel(frame), "data_by_year")
save_table(split.summary(frame), "split_summary")
save_table(pd.DataFrame({name: {"n_features": len(cols)} for name, cols in FEATURE_SETS.items()}).T,
           "feature_sets")

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.bar(by_year.index, by_year.filings, color="lightgray", label="filings")
ax1.set_ylabel("filings")
ax2 = ax1.twinx()
ax2.plot(by_year.index, 100 * by_year.positive_rate, "o-", color="tab:red", label="bankruptcy within 12m (%)")
ax2.set_ylabel("% of filings followed by bankruptcy")
for year, label in ((2019, "gap"), (2021, "gap")):
    ax1.axvspan(year - 0.5, year + 0.5, color="k", alpha=0.08)
ax1.set_title("Filings and 12-month bankruptcy rate by year (shaded = purged gap years)")
fig.legend(loc="upper right", fontsize=8)
plots._save(fig, FIG / "01_data_by_year.png");

## 2. Structural model

There are no daily stock prices for delisted firms, so the Merton model is fed with SEC data (see `src/structural/sec_structural.py`):

* **Accounting DD (`acct_dd`)**: Merton distance to default with book assets as asset value, the default point *current liabilities + ½ long-term liabilities*, asset volatility from the firm's own filing history (shrunk to its industry median), and drift from asset growth. This is the idea behind the KMV private-firm model.
* **Public-float naive DD (`pf_naive_dd`)**: the Bharath–Shumway (2008) naive DD, with the 10-K cover-page public float as market equity.

The **structural-only model** is a logistic map from these DDs to a PD, i.e. the Merton model used directly as a scoring model.

In [ ]:
dd_columns = {"acct_dd": "Accounting DD", "pf_naive_dd": "Public-float naive DD", "acct_d_va": "Default point / assets"}
plots.class_distributions(frame.loc[split.train], dd_columns, FIG / "02_dd_distributions.png")
plots.event_time_plot(frame, dd_columns, FIG / "02_dd_before_bankruptcy.png",
                      title="Structural signals in the 3 years before bankruptcy (bankrupt firms)");

**Structural-only model and DD variants (RQ3).** Each variant is fitted alone as a one-variable logit, so we can see how much the conclusions depend on how the DD is built: drift choice (`acct_dd_mu0` uses μ = 0) and default-point definition (`acct_dd_debt` uses short-term debt + ½ long-term debt, which is available for fewer firms).

In [ ]:
R = {}                                     # all main out-of-time results live here
prior = structural_prior(frame, split)     # structural log-odds for every row (used by H2)

R["structural_logit"] = fit_predict(MODELS["structural_logit"](), frame, feature_columns("structural"), split)

variants = {}
for column in ["acct_dd", "acct_dd_mu0", "acct_dd_debt", "acct_d_va", "pf_naive_dd", "pf_e_to_d"]:
    variants[column] = fit_predict(MODELS["structural_logit"](), frame, [column], split)
variant_table = results_table(variants)[["auroc", "average_precision", "ap_lift"]]
variant_table["coverage_test"] = [frame.loc[split.test, c].notna().mean() for c in variant_table.index]
save_table(variant_table, "structural_variants")

rank_corr = frame.loc[split.train, list(variants)].corr(method="spearman")
save_table(rank_corr, "structural_variant_rank_correlation")

## 3. Hyper-parameter tuning (equal budget)

Every model family gets the same budget: `TUNE_TRIALS` random-search trials, scored by mean average precision over **time-ordered folds inside the training years** (fit ≤ Y−2, score Y, for Y = 2015…2018). Moscatelli et al. (2020) use random 5-fold CV. That would leak here, because the same firm sits in many folds and neighbouring filings share an outcome window.

Each family is tuned once on the ML-only feature set, and those parameters are reused for its hybrid variants. This is conservative for the hybrids.

In [ ]:
PARAMS = {}
if TUNE_TRIALS:
    for family in ["logit", "random_forest", "xgboost", "xgboost_offset"]:
        spec = MODELS[family]()
        print(f"tuning {family}")
        best, trials = tune(spec, frame, feature_columns("ml"), split, n_trials=TUNE_TRIALS,
                            offset=prior if spec.supports_offset else None, verbose=False)
        PARAMS[family] = best
        trials.to_csv(TAB / f"tuning_{family}.csv", index=False)
        print(f"   best: {best}  (cv AP {trials.iloc[0]['cv_average_precision']:.4f})")

LSTM_PARAMS = {}
if RUN_LSTM:
    from src.pipeline.sequence import fit_predict_sequence, tune_sequence
    if LSTM_TUNE_TRIALS:
        LSTM_PARAMS, trials = tune_sequence(frame, feature_columns("ml"), split, n_trials=LSTM_TUNE_TRIALS)
        trials.to_csv(TAB / "tuning_lstm.csv", index=False)

(RESULTS / "params.json").write_text(json.dumps({"tabular": PARAMS, "lstm": LSTM_PARAMS}, indent=2, default=str))

In [ ]:
def run(name, family, feature_set, *, offset=None, results=R, data=frame, sp=split, **kwargs):
    """Fit one tabular model on a split, store and print its test scores."""
    r = fit_predict(MODELS[family](), data, feature_columns(feature_set), sp,
                    params=PARAMS.get(family, {}), offset=offset, **kwargs)
    results[name] = r
    print(f"{name:24s} test AUROC {r.metrics['test']['auroc']:.4f}   AP {r.metrics['test']['average_precision']:.4f}"
          f"   ({r.seconds:.0f}s)")
    return r


def run_lstm(name, feature_set, *, offset=None, results=R, **kwargs):
    r = fit_predict_sequence(frame, feature_columns(feature_set), split, params=LSTM_PARAMS,
                             seeds=LSTM_SEEDS, offset=offset, name=name, **kwargs)
    results[name] = r
    print(f"{name:24s} test AUROC {r.metrics['test']['auroc']:.4f}   AP {r.metrics['test']['average_precision']:.4f}"
          f"   ({r.seconds:.0f}s)")
    return r

## 4. ML-only models

No structural information. The feature set `ml` has 23 columns: balance-sheet ratios, income-statement and cash-flow ratios, the cover-page public float, and filing metadata.

| Model | What it is |
|---|---|
| `logit_ml` | Logistic regression (L2), the statistical benchmark (Moscatelli's LOG/PLR) |
| `rf_ml` | Random forest |
| `xgb_static_ml` | **XGB-static**: gradient-boosted trees on one filing |
| `xgb_lag_ml` | **XGB-lag**: the same, plus history features from the last 8 filings (changes, rolling mean/min/max/std, trend) |
| `lstm_ml` | LSTM over the last 8 filings (3-seed average) |

All use class weights for the ~1% positives and are then Platt-calibrated on the 2020 validation year.

In [ ]:
run("logit_ml", "logit", "ml")
run("rf_ml", "random_forest", "ml")
run("xgb_static_ml", "xgboost", "ml")
run("xgb_lag_ml", "xgboost", "ml_lag")
if RUN_LSTM:
    run_lstm("lstm_ml", "ml")

## 5. Hybrid models

* **H1, structure as features:** the same models on `hybrid` = `ml` + raw Merton inputs + structural outputs (DDs, D/V).
* **Control:** `xgb_static_raw_merton` gets the raw Merton *inputs* (default point, asset and equity volatility, drift) but not the DD formula. If H1 beats the control, the gain comes from the structural functional form, not from extra variables.
* **H2, structure as prior:** the model starts from the structural model's log-odds. For XGBoost they are passed as `base_margin`; for the LSTM they are added to the output logit. The ML part learns only a correction. With little data or in a new industry, the prediction should stay close to Merton.
* **H3, stacking:** a small logit combining the structural, XGBoost and LSTM predictions, fitted on the 2020 validation year.

In [ ]:
# H1: structural features
run("logit_hybrid", "logit", "hybrid")
run("rf_hybrid", "random_forest", "hybrid")
run("xgb_static_raw_merton", "xgboost", "ml_raw_merton")     # control
run("xgb_static_hybrid", "xgboost", "hybrid")
run("xgb_lag_hybrid", "xgboost", "hybrid_lag")
if RUN_LSTM:
    run_lstm("lstm_hybrid", "hybrid")

In [ ]:
# H2: structural prior
run("xgb_prior_ml", "xgboost_offset", "ml", offset=prior)
run("xgb_prior_lag", "xgboost_offset", "ml_lag", offset=prior)
if RUN_LSTM:
    run_lstm("lstm_prior_ml", "ml", offset=prior)

# H3: stacking
stack_members = {"static": ["structural_logit", "xgb_static_ml"], "history": ["structural_logit", "xgb_lag_ml"]}
if RUN_LSTM:
    stack_members = {k: v + ["lstm_ml"] for k, v in stack_members.items()}
for kind, members in stack_members.items():
    R[f"stack_{kind}"] = stack({m: R[m] for m in members}, name=f"stack_{kind}")
    print(f"stack_{kind:18s} test AUROC {R[f'stack_{kind}'].metrics['test']['auroc']:.4f}   "
          f"AP {R[f'stack_{kind}'].metrics['test']['average_precision']:.4f}   weights {R[f'stack_{kind}'].params['weights']}")

## 6. Main results: out-of-time test 2022–2025

* **AUROC**: ranking quality (0.5 = random).
* **Average precision (AP)**: area under the precision–recall curve. With a ~1% base rate this is the more demanding metric. *AP lift* = AP / base rate.
* **Brier** and **ECE**: probability accuracy after calibration.
* **Top-decile capture**: share of bankruptcies among the 10% riskiest filings.

Differences between models use a **paired firm bootstrap**: firms are resampled, not filings, because filings of one firm are correlated. A comparison is significant when its 95% interval excludes 0.

In [ ]:
main = results_table(R).sort_values("average_precision", ascending=False)
save_table(main, "main_results")

# predictions for later analysis
pd.concat([r.test.assign(experiment=name) for name, r in R.items()]).to_csv(PRED / "main_test_predictions.csv", index=False)

In [ ]:
COMPARISONS = [  # (A, B): is A better than B?
    ("xgb_static_ml", "structural_logit"),      # ML vs structural
    ("xgb_static_ml", "logit_ml"),              # trees vs statistical model (Moscatelli)
    ("rf_ml", "logit_ml"),
    ("logit_hybrid", "logit_ml"),               # H1 per model family (RQ1)
    ("rf_hybrid", "rf_ml"),
    ("xgb_static_hybrid", "xgb_static_ml"),
    ("xgb_static_hybrid", "xgb_static_raw_merton"),   # functional form vs raw inputs
    ("xgb_lag_hybrid", "xgb_lag_ml"),
    ("xgb_prior_ml", "xgb_static_ml"),          # H2
    ("xgb_prior_lag", "xgb_lag_ml"),
    ("xgb_lag_ml", "xgb_static_ml"),            # does history help? (RQ4)
    ("stack_static", "xgb_static_ml"),          # H3
]
if RUN_LSTM:
    COMPARISONS += [("lstm_ml", "xgb_lag_ml"), ("lstm_hybrid", "lstm_ml"), ("lstm_prior_ml", "lstm_ml")]

deltas = []
for a, b in COMPARISONS:
    d = paired_bootstrap_delta(R[a].test[TARGET], R[a].test["p"], R[b].test["p"], R[a].test["firm"],
                               replicates=BOOTSTRAP)
    deltas += [{"a": a, "b": b, "metric": metric, **row} for metric, row in d.to_dict("index").items()]
deltas = pd.DataFrame(deltas)
save_table(deltas.set_index(["a", "b", "metric"]), "main_deltas")
plots.delta_forest(deltas, FIG / "06_deltas_auroc.png", metric="auroc")
plots.delta_forest(deltas, FIG / "06_deltas_ap.png", metric="average_precision");

**The 2×2 design (RQ4).** Rows compare snapshot vs history and columns compare without vs with structure. Reading across a row shows whether structure helps; reading down a column shows whether history helps.

In [ ]:
def score(name, metric):
    return R[name].metrics["test"][metric] if name in R else np.nan

design = {"snapshot (XGB-static)": ("xgb_static_ml", "xgb_static_hybrid"),
          "history (XGB-lag)": ("xgb_lag_ml", "xgb_lag_hybrid"),
          "history (LSTM)": ("lstm_ml", "lstm_hybrid")}
for metric in ("auroc", "average_precision"):
    table = pd.DataFrame({row: {"no structure": score(a, metric), "with structure (H1)": score(b, metric)}
                          for row, (a, b) in design.items()}).T
    print(metric)
    save_table(table, f"two_by_two_{metric}")

In [ ]:
key = ["structural_logit", "logit_ml", "xgb_static_ml", "xgb_static_hybrid", "xgb_prior_ml", "xgb_lag_hybrid"]
key += ["lstm_ml", "stack_history"] if RUN_LSTM else ["stack_history"]
key_results = {k: R[k] for k in key if k in R}
plots.roc_pr_curves(key_results, FIG / "06_roc_pr.png", title="Out-of-time test 2022-2025")
plots.calibration_plot({k: R[k] for k in ["structural_logit", "xgb_static_ml", "xgb_static_hybrid"]},
                       FIG / "06_calibration.png");

print("mean predicted PD vs realised rate on test:")
display(pd.DataFrame({k: {"mean_pd": r.test["p"].mean(), "realised": r.test[TARGET].mean()}
                      for k, r in key_results.items()}).T.round(4))

**Backtesting by PD bucket** (Moscatelli et al. 2020, Table 6). Test filings are grouped by predicted PD into credit-quality steps. Each bucket's realised bankruptcy rate is compared with its upper PD limit using a binomial test. Green means consistent; yellow and red mean more bankruptcies than the bucket allows. Filings of one firm are correlated, so read the colours as indicative.

In [ ]:
for name in ["structural_logit", "xgb_static_ml", "xgb_static_hybrid"]:
    print(name)
    save_table(backtest_buckets(R[name].test[TARGET], R[name].test["p"]), f"backtest_{name}")

## 7. What drives the predictions

* **Permutation importance** (Moscatelli Table 9): the drop in test AUROC when one feature is randomly shuffled, for the hybrid XGBoost.
* **AUROC by sector and size** (Moscatelli Table 5): where do the models work, and where does structure help most? Groups with fewer than 10 bankrupt filings in the test set are skipped.

In [ ]:
xgb_h = fit_predict(MODELS["xgboost"](), frame, feature_columns("hybrid"), split,
                    params=PARAMS.get("xgboost", {}), keep_model=True)
importance = permutation_importance(xgb_h, frame, split, n_repeats=3)
save_table(importance.head(20), "permutation_importance_xgb_hybrid")
plots.horizontal_bars(importance.head(20), "feature", "auroc_drop", FIG / "07_permutation_importance.png",
                      error="std", title="Permutation importance, XGB hybrid", xlabel="drop in test AUROC");

In [ ]:
def auroc_by_group(names, group_column):
    rows = []
    for name in names:
        t = R[name].test
        for group, g in t.groupby(group_column, observed=True):
            if g[TARGET].sum() >= 10:
                rows.append({"group": group, "model": name, "auroc": roc_auc_score(g[TARGET], g["p"]),
                             "positives": int(g[TARGET].sum()), "rows": len(g)})
    return pd.DataFrame(rows)

compare = ["structural_logit", "xgb_static_ml", "xgb_static_hybrid", "xgb_prior_ml"]
for name in compare:   # size terciles of total assets within the test period
    R[name].test["size"] = pd.qcut(R[name].test["log_assets_usd"], 3, labels=["small", "medium", "large"])

by_sector = save_table(auroc_by_group(compare, "sic_division").set_index(["group", "model"]), "auroc_by_sector")
by_size = save_table(auroc_by_group(compare, "size").set_index(["group", "model"]), "auroc_by_size")
plots.grouped_bars(by_sector.reset_index(), "group", "auroc", "model", FIG / "07_auroc_by_sector.png",
                   title="Test AUROC by SIC division", ylim=(0.5, 1))
plots.grouped_bars(by_size.reset_index(), "group", "auroc", "model", FIG / "07_auroc_by_size.png",
                   title="Test AUROC by firm size", ylim=(0.5, 1));

## 8. Evaluation regimes (RQ2: does structure help more when the world shifts?)

Each regime compares the same four models:

| Label | Model |
|---|---|
| Structural | DD-only logit |
| ML (XGB) | XGBoost on `ml` |
| Hybrid H1 (XGB) | XGBoost on `hybrid` |
| Hybrid H2 (XGB + prior) | XGBoost on `ml`, starting from the structural prior |

Default hyper-parameters are used here, because the tuned ones were chosen with data up to 2018, which overlaps the test periods of some regimes.

In [ ]:
REGIME_MODELS = {
    "Structural": ("structural_logit", "structural", False),
    "ML (XGB)": ("xgboost", "ml", False),
    "Hybrid H1 (XGB)": ("xgboost", "hybrid", False),
    "Hybrid H2 (XGB + prior)": ("xgboost_offset", "ml", True),
}


def run_regime(data, sp, **info):
    """Fit the four regime models on one split; one row of test metrics per model."""
    regime_prior = structural_prior(data, sp)     # refit on this split's training rows
    rows = []
    for label, (family, feature_set, use_prior) in REGIME_MODELS.items():
        r = fit_predict(MODELS[family](), data, feature_columns(feature_set), sp,
                        offset=regime_prior if use_prior else None)
        m = r.metrics["test"]
        rows.append({**info, "model": label, "auroc": m["auroc"], "ap": m["average_precision"],
                     "ap_lift": m["ap_lift"], "test_rows": m["rows"], "test_positives": m["positives"],
                     "train_positive_firms": data.loc[sp.train & data[TARGET].eq(1).to_numpy(), "firm"].nunique()})
    return pd.DataFrame(rows)

### 8.1 Rolling out-of-time, year by year

For each test year T: train ≤ T−4, validate T−2, test T. Shows whether performance holds up across the cycle, including the 2015–16 commodity bust, COVID (2020) and the 2022–23 rate rises.

In [ ]:
rolling = pd.concat([run_regime(frame, sp, test_year=sp.info["test_start_year"])
                     for sp in rolling_splits(frame, range(2015, 2026))], ignore_index=True)
save_table(rolling.set_index(["test_year", "model"]), "regime_rolling_years")
plots.line_plot(rolling, "test_year", "auroc", "model", FIG / "08_rolling_auroc.png", title="AUROC by test year")
plots.line_plot(rolling, "test_year", "ap_lift", "model", FIG / "08_rolling_ap_lift.png",
                title="AP lift (AP / base rate) by test year", ylabel="AP lift");

### 8.2 Out-of-industry and out-of-industry + out-of-time

One SIC division is held out at a time, and models train on all other divisions. Two settings:
* **Out-of-industry:** test on the held-out industry in the training years.
* **Out-of-industry + out-of-time:** test on the held-out industry in 2022–25. This is the hardest case.

Only divisions with at least 20 bankrupt firms are held out.

In [ ]:
positive_firms = frame[frame[TARGET].eq(1)].groupby("sic_division").firm.nunique().sort_values(ascending=False)
held_out_divisions = positive_firms[positive_firms >= 20].index.tolist()
print("held-out divisions:", dict(positive_firms[held_out_divisions]))

ooi = []
for division in held_out_divisions:
    for out_of_time in (False, True):
        try:
            sp = industry_split(frame, division, out_of_time=out_of_time)
        except ValueError as err:          # e.g. no bankruptcies of that industry in the test years
            print("skipped:", err)
            continue
        setting = "out-of-industry + out-of-time" if out_of_time else "out-of-industry"
        ooi.append(run_regime(frame, sp, division=division, setting=setting))
ooi = pd.concat(ooi, ignore_index=True)
save_table(ooi.set_index(["setting", "division", "model"]), "regime_out_of_industry")
for setting, g in ooi.groupby("setting"):
    name = setting.replace(" ", "_").replace("+", "and")
    plots.grouped_bars(g, "division", "auroc", "model", FIG / f"08_{name}.png", title=f"{setting}: AUROC",
                       ylim=(0.5, 1))
save_table(ooi.groupby(["setting", "model"])[["auroc", "ap_lift"]].mean(), "regime_out_of_industry_mean");

### 8.3 Unseen firms

30% of firms are held out completely, and the test set contains only them in 2022–25. This removes the advantage of having seen a firm's earlier filings in training. Repeated with three random firm draws.

In [ ]:
unseen = pd.concat([run_regime(frame, firm_holdout_split(frame, fraction=0.3, seed=s), seed=s) for s in range(3)],
                   ignore_index=True)
save_table(unseen.groupby("model")[["auroc", "ap", "ap_lift"]].agg(["mean", "std"]), "regime_unseen_firms");

### 8.4 Limited data

Training uses only a random fraction of the training firms (validation and test unchanged), repeated `LIMITED_DATA_REPEATS` times. The structural prior should matter most here. Moscatelli et al. found that the ML advantage disappears on small samples. Fractions leaving fewer than 10 bankrupt training firms are skipped.

In [ ]:
fractions = [0.02, 0.05, 0.10, 0.25, 0.50, 1.0]
limited = []
for fraction in fractions:
    for seed in range(1 if fraction == 1.0 else LIMITED_DATA_REPEATS):
        sp = subsample_training_firms(frame, split, fraction=fraction, seed=seed)
        n_pos = frame.loc[sp.train & frame[TARGET].eq(1).to_numpy(), "firm"].nunique()
        if n_pos < 10:
            continue
        limited.append(run_regime(frame, sp, fraction=fraction, seed=seed))
limited = pd.concat(limited, ignore_index=True)
summary = limited.groupby(["fraction", "model"]).agg(auroc=("auroc", "mean"), auroc_sd=("auroc", "std"),
                                                    ap=("ap", "mean"), ap_sd=("ap", "std"),
                                                    train_positive_firms=("train_positive_firms", "mean")).reset_index()
summary[["auroc_sd", "ap_sd"]] = summary[["auroc_sd", "ap_sd"]].fillna(0)
save_table(summary.set_index(["fraction", "model"]), "regime_limited_data")
for metric in ("auroc", "ap"):
    s = summary.assign(low=summary[metric] - summary[f"{metric}_sd"], high=summary[metric] + summary[f"{metric}_sd"])
    plots.line_plot(s, "fraction", metric, "model", FIG / f"08_limited_data_{metric}.png", band=("low", "high"),
                    logx=True, title=f"Test {metric.upper()} vs share of training firms (mean ± sd)");

### 8.5 Prediction horizon

The label is redefined as bankruptcy within 3, 6 or 12 months of the filing, and all models are refitted. Shorter horizons should be easier. The question is whether structure's contribution changes with the horizon.

In [ ]:
horizon = []
for months in (3, 6, 12):
    relabelled = relabel(frame, months)
    horizon.append(run_regime(relabelled, oot_split(relabelled), horizon_months=months))
horizon = pd.concat(horizon, ignore_index=True)
save_table(horizon.set_index(["horizon_months", "model"]), "regime_horizon")
plots.line_plot(horizon, "horizon_months", "auroc", "model", FIG / "08_horizon_auroc.png",
                title="AUROC by prediction horizon")
plots.line_plot(horizon, "horizon_months", "ap_lift", "model", FIG / "08_horizon_ap_lift.png",
                title="AP lift by prediction horizon", ylabel="AP lift");

## 9. Ablation studies

All on the main out-of-time split with XGBoost.

### 9.1 Feature-group ablation

Start from the full `hybrid` feature set and drop one group at a time. The fall in test AUROC/AP shows how much each group contributes *given all the others*.

In [ ]:
hybrid_groups = ["accounting_base", "accounting_flow", "market_cover_page", "filing_metadata",
                 "merton_inputs", "structural_dd"]
full = R["xgb_static_hybrid"].metrics["test"]
ablation = []
for group in hybrid_groups:
    kept = [g for g in hybrid_groups if g != group]
    r = fit_predict(MODELS["xgboost"](), frame, feature_columns(kept), split, params=PARAMS.get("xgboost", {}))
    m = r.metrics["test"]
    ablation.append({"dropped_group": group, "n_dropped": len(GROUPS[group]),
                     "auroc_drop": full["auroc"] - m["auroc"], "ap_drop": full["average_precision"] - m["average_precision"]})
ablation = pd.DataFrame(ablation).sort_values("auroc_drop", ascending=False)
save_table(ablation.set_index("dropped_group"), "ablation_feature_groups")
plots.horizontal_bars(ablation, "dropped_group", "auroc_drop", FIG / "09_ablation_groups.png",
                      title="Drop in test AUROC when a group is removed (XGB hybrid)", xlabel="AUROC drop");

### 9.2 Which structural variant to use inside the hybrid (RQ3)

The hybrid is refitted with the accounting DD swapped for its variants (no drift; debt-based default point), and with only one of the two DD families.

In [ ]:
base = feature_columns("hybrid")
swap = lambda old, new: [new if c == old else c for c in base]
variants_h = {
    "acct_dd (main)": base,
    "acct_dd_mu0 (no drift)": swap("acct_dd", "acct_dd_mu0"),
    "acct_dd_debt (debt default point)": swap("acct_dd", "acct_dd_debt"),
    "accounting DD only": [c for c in base if c not in ("pf_naive_dd", "pf_e_to_d")],
    "public-float DD only": [c for c in base if c not in ("acct_dd", "acct_d_va")],
}
rows = {}
for label, columns in variants_h.items():
    m = fit_predict(MODELS["xgboost"](), frame, columns, split, params=PARAMS.get("xgboost", {})).metrics["test"]
    rows[label] = {"auroc": m["auroc"], "ap": m["average_precision"], "ap_lift": m["ap_lift"]}
save_table(pd.DataFrame(rows).T, "ablation_structural_variants");

### 9.3 Rare-event handling: class weights vs downsampling + Bayes correction

Moscatelli et al. train on a balanced, downsampled sample and map the PDs back with the Bayes correction (their eq. A2). We compare this with class weights and with no adjustment. The uncalibrated mean PD shows whether the correction alone recovers the right PD level, before validation calibration.

In [ ]:
rows = {}
for label, imbalance in [("class weights", "weights"), ("downsample + Bayes", "downsample"), ("none", "none")]:
    spec = replace(MODELS["xgboost"](), imbalance=imbalance)
    r = fit_predict(spec, frame, feature_columns("hybrid"), split, params=PARAMS.get("xgboost", {}))
    rows[label] = {"auroc": r.metrics["test"]["auroc"], "ap": r.metrics["test"]["average_precision"],
                   "brier_calibrated": r.metrics["test"]["brier"],
                   "brier_uncalibrated": r.metrics["test_uncalibrated"]["brier"],
                   "mean_pd_uncalibrated": r.test["p_raw"].mean(), "mean_pd_calibrated": r.test["p"].mean(),
                   "realised_rate": r.test[TARGET].mean()}
save_table(pd.DataFrame(rows).T, "ablation_imbalance");

### 9.4 Calibration method

Platt scaling vs isotonic regression vs no calibration. Ranking (AUROC) barely changes; Brier, ECE and the PD level do. The calibration year (2020) had an unusually low bankruptcy rate, so all calibrated PDs tend to be too low in 2022–25. This is the "slow adaptation" problem Moscatelli et al. report for 2012 and 2014.

In [ ]:
rows = {}
for method in ("platt", "isotonic", "none"):
    r = fit_predict(MODELS["xgboost"](), frame, feature_columns("hybrid"), split, params=PARAMS.get("xgboost", {}),
                    calibration=method)
    m = r.metrics["test"]
    rows[method] = {"auroc": m["auroc"], "brier": m["brier"], "ece": m["ece"], "log_loss": m["log_loss"],
                    "mean_pd": r.test["p"].mean(), "realised_rate": r.test[TARGET].mean()}
save_table(pd.DataFrame(rows).T, "ablation_calibration");

### 9.5 LSTM history length

How many past filings should the LSTM read? Window lengths of 4, 8 and 12 filings (about 1, 2 and 3 years), with a single seed to save time.

In [ ]:
if RUN_LSTM:
    rows = {}
    for window in (4, 8, 12):
        r = fit_predict_sequence(frame, feature_columns("ml"), split, params={**LSTM_PARAMS, "window": window},
                                 seeds=(0,), name=f"lstm_w{window}")
        rows[window] = {"auroc": r.metrics["test"]["auroc"], "ap": r.metrics["test"]["average_precision"],
                        "seconds": r.seconds}
    save_table(pd.DataFrame(rows).T.rename_axis("window_filings"), "ablation_lstm_window")

## 10. Summary of saved outputs

Everything needed for the report is in `results/`:
* `tables/main_results.csv` and `tables/main_deltas.csv`: the headline comparison (RQ1, RQ4)
* `tables/regime_*.csv` and `figures/08_*`: rolling years, out-of-industry, unseen firms, limited data, horizon (RQ2)
* `tables/structural_variants.csv` and `tables/ablation_structural_variants.csv`: DD construction (RQ3)
* `tables/ablation_*.csv`: feature groups, imbalance handling, calibration, LSTM window
* `predictions/main_test_predictions.csv`: test-set PDs of every main model
* `params.json`: tuned hyper-parameters

In [ ]:
for path in sorted(RESULTS.rglob("*")):
    if path.is_file():
        print(f"{path.stat().st_size / 1024:9.1f} KB  {path}")